# Exp 9: Covariance-Matched Random Vectors — Batch 5/7 (Vectors 13 to 15)
Evaluates Covariance-Matched vectors 13..15 on 500 test samples (max_new_tokens=200, greedy).
Runtime: ~8.7 hours total (completes 100% cleanly within Kaggle 12h limit).

In [ ]:
!pip install -q bitsandbytes accelerate transformers torch bert-score tqdm scikit-learn
print('Dependencies installed!')

In [ ]:
import os, json, glob, random, time, math, gc, datetime
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BEST_LAYER = 8
ALPHA_0 = 18.0
K = 16
DECAY = 'linear'
MAX_NEW_TOKENS = 200
N_COV_MATCHED = 20
LOG_EVERY = 25
CHECKPOINT_EVERY = 100

OUTPUT_DIR = '/kaggle/working'
print(f'Config: Layer={BEST_LAYER}, a0={ALPHA_0}, K={K}, decay={DECAY}, tokens={MAX_NEW_TOKENS}')
print(f'Logging every {LOG_EVERY} samples, checkpoint every {CHECKPOINT_EVERY} samples')

In [ ]:
# Load Dataset & Split
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}', f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}', f'./{DATA_FILENAME}',
    f'E:/Paper_Steering_VN_15K/data/{DATA_FILENAME}'
]
data_path = None
for p in search_paths:
    m = glob.glob(p, recursive=True)
    if m: data_path = m[0]; break
if not data_path: raise FileNotFoundError(f'{DATA_FILENAME} not found')
print(f'Dataset: {data_path}')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

shuffled_records = list(raw_dataset)
random.seed(SEED); random.shuffle(shuffled_records)
n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
train_records = shuffled_records[:n_train]
test_records = shuffled_records[n_train + n_val:]
test_subset = test_records[:500]
print(f'Train: {len(train_records)}, Test total: {len(test_records)}, Eval subset: {len(test_subset)}')

In [ ]:
# Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Loading {MODEL_NAME}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
if torch.cuda.is_available():
    print(f'GPU Memory after load: {torch.cuda.memory_allocated()/(1024**3):.2f} GB')
print('Model loaded!')

In [ ]:
# Extract Training Activations
PROMPT_TEMPLATE = """Dua vao ngu canh y hoc sau day, hay tra loi cau hoi:
Ngu canh: {context}
Cau hoi: {question}
Tra loi: """

probe_count = min(400, len(train_records))
probe_records = train_records[:probe_count]
print(f'Extracting Layer {BEST_LAYER} activations from {probe_count} pairs...')

activations_pos = []
activations_neg = []
t0 = time.time()

with torch.no_grad():
    for i, rec in enumerate(probe_records):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        text_pos = PROMPT_TEMPLATE.format(context=ctx, question=q) + rec['right_answer']
        text_neg = PROMPT_TEMPLATE.format(context=ctx, question=q) + rec['hallucinated_answer']
        
        inp_p = tokenizer(text_pos, return_tensors='pt', padding=False).to('cuda')
        inp_n = tokenizer(text_neg, return_tensors='pt', padding=False).to('cuda')
        out_p = model(**inp_p, output_hidden_states=True)
        out_n = model(**inp_n, output_hidden_states=True)
        
        activations_pos.append(out_p.hidden_states[BEST_LAYER][0, -1, :].detach().float().cpu().numpy())
        activations_neg.append(out_n.hidden_states[BEST_LAYER][0, -1, :].detach().float().cpu().numpy())
        
        if (i+1) % 50 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (i+1) * (probe_count - i - 1)
            print(f'  Activation extraction: {i+1}/{probe_count} | '
                  f'Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')

activations_pos = np.array(activations_pos)
activations_neg = np.array(activations_neg)
print(f'Extracted: pos={activations_pos.shape}, neg={activations_neg.shape} in {(time.time()-t0)/60:.1f}min')

In [ ]:
# Compute Original v_steer
mean_pos = np.mean(activations_pos, axis=0)
mean_neg = np.mean(activations_neg, axis=0)
v_raw = mean_pos - mean_neg
v_steer_np = v_raw / np.linalg.norm(v_raw)
v_steer = torch.tensor(v_steer_np, dtype=torch.bfloat16, device='cuda')
print(f'Original v_steer: norm={torch.norm(v_steer).item():.4f}, dim={v_steer.shape[0]}')

In [ ]:
# Compute Covariance-Matched Random Vectors for Batch 5 (Vectors 13 to 15)
print('Computing covariance-matched random vectors for Batch 5...')
diff_vectors = activations_pos - activations_neg
from sklearn.covariance import LedoitWolf
print('  Fitting Ledoit-Wolf shrinkage covariance estimator...')
lw = LedoitWolf()
lw.fit(diff_vectors)
cov_matrix = lw.covariance_
print(f'  Covariance matrix: {cov_matrix.shape}, shrinkage={lw.shrinkage_:.4f}')
try:
    L = np.linalg.cholesky(cov_matrix + 1e-6 * np.eye(cov_matrix.shape[0]))
except np.linalg.LinAlgError:
    eigvals, eigvecs = np.linalg.eigh(cov_matrix)
    eigvals = np.maximum(eigvals, 1e-6)
    L = eigvecs @ np.diag(np.sqrt(eigvals))

cov_matched_vectors = []
cos_sims_cov = []
V_START = 12
V_END = 15
BATCH_IDX = 5
for i in range(V_START, V_END):
    np.random.seed(SEED + 3000 + i)
    z = np.random.randn(cov_matrix.shape[0])
    v_cov = L @ z
    v_cov_norm = v_cov / np.linalg.norm(v_cov)
    cov_matched_vectors.append(torch.tensor(v_cov_norm, dtype=torch.bfloat16, device='cuda'))
    cs = np.dot(v_steer_np, v_cov_norm)
    cos_sims_cov.append(cs)
    print(f'  CovMatched-{i+1:02d}: cos(v_steer, v_cov)={cs:.4f}')

print(f'Batch 5: {len(cov_matched_vectors)} vectors computed (seeds {SEED+3000+V_START} to {SEED+3000+V_END-1})')


In [ ]:
# === EVALUATION ENGINE WITH DETAILED LOGGING ===
from bert_score import score as bert_score_fn

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=18.0, K=16, decay='linear'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    def _eff_alpha(self, t):
        if self.K >= 999: return self.alpha
        if t >= self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'linear': return self.alpha * (1.0 - t / self.K)
        return self.alpha
    def hook_fn(self, module, inputs, output):
        a = self._eff_alpha(self.step_counter)
        if a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + a * v
        self.step_counter += 1
        return output
    def register(self, mdl):
        self.step_counter = 0
        self.handle = mdl.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    def remove(self):
        if self.handle: self.handle.remove(); self.handle = None

def generate_outputs(model, tokenizer, records, v_vector, alpha, K, decay,
                     max_new_tokens=200, layer_idx=8, name=''):
    gen_texts, ref_ans, neg_ans = [], [], []
    latencies, token_counts = [], []
    total_n = len(records)
    t_start = time.time()
    
    print(f'\n{"="*70}')
    print(f'[{name}] START: {total_n} samples, max_tokens={max_new_tokens}')
    print(f'[{name}] alpha={alpha}, K={K}, decay={decay}, layer={layer_idx}')
    if torch.cuda.is_available():
        print(f'[{name}] GPU: {torch.cuda.memory_allocated()/(1024**3):.2f} GB used')
    print(f'[{name}] Time: {datetime.datetime.now().strftime("%H:%M:%S")}')
    print(f'{"="*70}')
    
    for idx, rec in enumerate(records):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        hook = None
        if v_vector is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False, temperature=1.0, top_p=1.0)
        lat = time.time() - t0
        if hook: hook.remove()
        
        gen_tok = out_ids[0][inputs.input_ids.shape[1]:]
        gen_texts.append(tokenizer.decode(gen_tok, skip_special_tokens=True))
        ref_ans.append(rec['right_answer'])
        neg_ans.append(rec['hallucinated_answer'])
        latencies.append(lat)
        token_counts.append(len(gen_tok))
        
        # Log every LOG_EVERY samples
        if (idx+1) % LOG_EVERY == 0 or idx == total_n - 1:
            elapsed = time.time() - t_start
            speed = (idx+1) / elapsed
            eta = (total_n - idx - 1) / speed if speed > 0 else 0
            gpu_gb = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
            print(f'  [{name}] {idx+1:>4}/{total_n} | '
                  f'{elapsed/60:.1f}min elapsed | ETA {eta/60:.1f}min | '
                  f'{speed:.2f} samp/s | lat {np.mean(latencies[-LOG_EVERY:]):.2f}s | '
                  f'tok {np.mean(token_counts[-LOG_EVERY:]):.0f} | GPU {gpu_gb:.2f}GB')
        
        # Checkpoint every CHECKPOINT_EVERY samples
        if (idx+1) % CHECKPOINT_EVERY == 0:
            safe_name = name.replace(' ', '_').replace('/', '_')
            ckpt = {'name': name, 'done': idx+1, 'total': total_n,
                    'texts': gen_texts.copy(), 'refs': ref_ans.copy(), 'negs': neg_ans.copy()}
            ckpt_path = os.path.join(OUTPUT_DIR, f'ckpt_{safe_name}_{idx+1}.json')
            with open(ckpt_path, 'w', encoding='utf-8') as f:
                json.dump(ckpt, f, ensure_ascii=False)
            print(f'  [{name}] >>> CHECKPOINT SAVED: {ckpt_path}')
    
    gen_time = time.time() - t_start
    print(f'\n  [{name}] Generation done: {gen_time/60:.1f}min ({gen_time/total_n:.2f}s/sample)')
    
    # BERTScore
    print(f'  [{name}] Computing BERTScore (batch, verbose)...')
    t_bs = time.time()
    _, _, bs_ref = bert_score_fn(gen_texts, ref_ans, lang='vi',
                                  model_type='bert-base-multilingual-cased', num_layers=9,
                                  verbose=True, batch_size=64)
    _, _, bs_neg = bert_score_fn(gen_texts, neg_ans, lang='vi',
                                  model_type='bert-base-multilingual-cased', num_layers=9,
                                  verbose=True, batch_size=64)
    bs_time = time.time() - t_bs
    print(f'  [{name}] BERTScore done: {bs_time/60:.1f}min')
    
    bs_r, bs_n = bs_ref.numpy(), bs_neg.numpy()
    rp = (bs_r > bs_n).astype(int)
    rp_pct = rp.mean() * 100
    rp_cnt = int(rp.sum())
    bs_f1 = float(np.mean(bs_r))
    
    print(f'\n  ========== RESULT [{name}] ==========')
    print(f'  RefPref: {rp_pct:.2f}% ({rp_cnt}/{len(rp)})')
    print(f'  BERTScore F1: {bs_f1:.4f}')
    print(f'  Avg tokens: {np.mean(token_counts):.1f}, Avg latency: {np.mean(latencies):.2f}s')
    print(f'  Total time: {(gen_time+bs_time)/60:.1f}min')
    print(f'  ======================================')
    
    return {'name': name, 'refpref_pct': rp_pct, 'refpref_count': rp_cnt,
            'total': len(rp), 'bertscore_f1': bs_f1, 'ref_pref_array': rp.tolist(),
            'gen_time_min': gen_time/60, 'bs_time_min': bs_time/60,
            'avg_latency': float(np.mean(latencies)), 'avg_tokens': float(np.mean(token_counts))}

print('Engine ready (with detailed logging + checkpoints).')

In [ ]:
# === RUN BATCH 5: COVARIANCE MATCHED VECTORS 13..15 ===
print('\n' + '#'*70)
print('# BATCH 5: COVARIANCE MATCHED VECTORS 13..15')
print('#'*70)
results_cov = []
for idx_in_batch, v_c in enumerate(cov_matched_vectors):
    abs_v_idx = V_START + idx_in_batch + 1
    print(f'\n--- Running Covariance-Matched Vector {abs_v_idx}/20 ---')
    r = generate_outputs(model, tokenizer, test_subset,
        v_vector=v_c, alpha=ALPHA_0, K=K, decay=DECAY,
        max_new_tokens=MAX_NEW_TOKENS, layer_idx=BEST_LAYER,
        name=f'CovMatch_{abs_v_idx:02d}')
    results_cov.append(r)
    gc.collect(); torch.cuda.empty_cache()
    running_rps = [x['refpref_pct'] for x in results_cov]
    print(f'  Batch 5 running avg ({len(results_cov)}/{len(cov_matched_vectors)}): RefPref={np.mean(running_rps):.2f}%')


In [ ]:
# === SUMMARY & SAVE BATCH 5 ===
cov_rps = [r['refpref_pct'] for r in results_cov]
cov_bs = [r['bertscore_f1'] for r in results_cov]
print('\n' + '='*80)
print('COVARIANCE MATCHED BATCH 5 SUMMARY')
print('='*80)
for r in results_cov:
    print(f"{r['name']:<20} RefPref: {r['refpref_pct']:.2f}% | BS-F1: {r['bertscore_f1']:.4f}")
print(f'Batch Mean RefPref: {np.mean(cov_rps):.2f}% +/- {np.std(cov_rps):.2f}%')

batch_summary = {
    'batch_idx': 5,
    'v_start': V_START + 1, 'v_end': V_END,
    'results': results_cov,
    'mean_refpref': float(np.mean(cov_rps)),
    'std_refpref': float(np.std(cov_rps)),
    'mean_bertscore_f1': float(np.mean(cov_bs)),
    'cosines': [float(c) for c in cos_sims_cov]
}
out_path = os.path.join(OUTPUT_DIR, 'exp09_covmatch_batch5.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(batch_summary, f, indent=2, ensure_ascii=False)
print(f'\nSaved: {out_path}')
